# Comparative Analysis: 3 RAG Systems

**Purpose**: Compare three different RAG system architectures:
1. **O-RAG (Current)** - Qwen 2.5 1.5B + Nomic embeddings with memory optimization
2. **Dense Retrieval RAG** - Pure vector similarity without reranking
3. **Hybrid RAG** - BM25 + Vector combination with reranking

**Metrics**:
- Query latency
- Retrieval quality (relevance scores)
- Generation quality (semantic coherence)
- Memory efficiency
- Throughput (queries/second)

## 1. Setup and Imports

In [ ]:
import os
import sys
import time
import json
import psutil
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML

sys.path.insert(0, os.path.abspath('..'))

from rag.pipeline import RAGPipeline
from rag.llm import LlamaCppModel
from rag.retriever import VectorStoreRetriever
from rag.memory_manager import MemoryManager, get_memory_manager

pd.set_option('display.max_colwidth', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')

print("✅ All imports successful")

## 2. Define Test Queries and Expected Relevance

In [ ]:
# Test dataset with expected relevance profiles
TEST_QUERIES = [
    {
        'query': 'What is retrieval augmented generation and how does it work?',
        'expected_relevance': 'high',
        'category': 'technical'
    },
    {
        'query': 'Compare different embedding models for semantic search',
        'expected_relevance': 'high',
        'category': 'ml'
    },
    {
        'query': 'How to optimize models for mobile devices',
        'expected_relevance': 'medium',
        'category': 'mobile'
    },
    {
        'query': 'Memory management techniques in Python',
        'expected_relevance': 'medium',
        'category': 'python'
    },
    {
        'query': 'What is the weather today',
        'expected_relevance': 'low',
        'category': 'out_of_domain'
    },
    {
        'query': 'How do attention mechanisms work in transformers?',
        'expected_relevance': 'high',
        'category': 'ml'
    },
    {
        'query': 'Difference between supervised and unsupervised learning',
        'expected_relevance': 'high',
        'category': 'ml'
    },
    {
        'query': 'Explain quantum computing basics',
        'expected_relevance': 'low',
        'category': 'out_of_domain'
    }
]

print(f"📋 Test Dataset: {len(TEST_QUERIES)} queries")
for i, q in enumerate(TEST_QUERIES, 1):
    print(f"  {i}. [{q['category']:12}] {q['query'][:50]}...")

## 3. RAG System 1: O-RAG (Current Implementation)

In [ ]:
print("\n🔄 Initializing O-RAG System...")
start_init = time.time()

try:
    rag_orag = RAGPipeline(
        model_path="qwen2.5-1.5b-instruct-q4_k_m.gguf",
        embedding_model_path="nomic-embed-text-v1.5.Q4_K_M.gguf",
        context_window=512,
        max_tokens=256,
        temperature=0.7,
        top_k=5
    )
    orag_init_time = time.time() - start_init
    print(f"✅ O-RAG initialized in {orag_init_time:.2f}s")
    orag_available = True
except Exception as e:
    print(f"⚠️ O-RAG unavailable: {e}")
    orag_available = False

## 4. RAG System 2: Dense Vector Retrieval (Baseline)

In [ ]:
print("\n🔄 Initializing Dense Retrieval RAG...")
start_init = time.time()

try:
    rag_dense = RAGPipeline(
        model_path="qwen2.5-1.5b-instruct-q4_k_m.gguf",
        embedding_model_path="nomic-embed-text-v1.5.Q4_K_M.gguf",
        context_window=256,  # Reduced context
        max_tokens=128,      # Reduced output
        temperature=0.5,
        top_k=3              # Fewer retrieved docs
    )
    dense_init_time = time.time() - start_init
    print(f"✅ Dense Retrieval initialized in {dense_init_time:.2f}s")
    dense_available = True
except Exception as e:
    print(f"⚠️ Dense Retrieval unavailable: {e}")
    dense_available = False

## 5. RAG System 3: Lightweight RAG (Mobile-Optimized)

In [ ]:
print("\n🔄 Initializing Mobile-Optimized RAG...")
start_init = time.time()

try:
    rag_mobile = RAGPipeline(
        model_path="qwen2.5-1.5b-instruct-q4_k_m.gguf",
        embedding_model_path="nomic-embed-text-v1.5.Q4_K_M.gguf",
        context_window=128,  # Minimal context (4GB device)
        max_tokens=64,       # Minimal output
        temperature=0.3,
        top_k=2              # Minimal retrieval
    )
    mobile_init_time = time.time() - start_init
    print(f"✅ Mobile-Optimized initialized in {mobile_init_time:.2f}s")
    mobile_available = True
except Exception as e:
    print(f"⚠️ Mobile-Optimized unavailable: {e}")
    mobile_available = False

# Summary
print("\n" + "="*70)
print("RAG Systems Available:")
print(f"  • O-RAG (Full-Featured): {'✅' if orag_available else '❌'}")
print(f"  • Dense Vector RAG: {'✅' if dense_available else '❌'}")
print(f"  • Mobile-Optimized RAG: {'✅' if mobile_available else '❌'}")

## 6. Execute Queries on All Systems

In [ ]:
# Results collection
comparison_results = []

print("\n🚀 Executing Comparative Benchmark...")
print("="*100)

for q_idx, query_obj in enumerate(TEST_QUERIES, 1):
    query = query_obj['query']
    category = query_obj['category']
    expected_rel = query_obj['expected_relevance']
    
    print(f"\n📝 Query {q_idx}: {query[:50]}...")
    print(f"   Category: {category} | Expected Relevance: {expected_rel}")
    print("-" * 100)
    
    # System 1: O-RAG
    if orag_available:
        try:
            start = time.time()
            start_mem = psutil.virtual_memory().available / (1024**2)
            
            response = rag_orag.ask(query, top_k=5)
            
            end = time.time()
            end_mem = psutil.virtual_memory().available / (1024**2)
            latency = (end - start) * 1000
            
            comparison_results.append({
                'query_id': q_idx,
                'query': query,
                'category': category,
                'system': 'O-RAG',
                'context_window': 512,
                'top_k': 5,
                'latency_ms': latency,
                'memory_available_mb': end_mem,
                'response_length': len(response.get('answer', '')),
                'retrieved_count': len(response.get('retrieved_docs', [])),
                'confidence': response.get('confidence', 0),
                'status': 'success'
            })
            print(f"   O-RAG: ✅ {latency:.0f}ms | {response.get('answer', '')[:50]}...")
        except Exception as e:
            print(f"   O-RAG: ❌ {str(e)[:50]}")
            comparison_results.append({
                'query_id': q_idx,
                'query': query,
                'category': category,
                'system': 'O-RAG',
                'context_window': 512,
                'top_k': 5,
                'latency_ms': 0,
                'memory_available_mb': 0,
                'response_length': 0,
                'retrieved_count': 0,
                'confidence': 0,
                'status': 'error'
            })
    
    # System 2: Dense Vector
    if dense_available:
        try:
            start = time.time()
            start_mem = psutil.virtual_memory().available / (1024**2)
            
            response = rag_dense.ask(query, top_k=3)
            
            end = time.time()
            end_mem = psutil.virtual_memory().available / (1024**2)
            latency = (end - start) * 1000
            
            comparison_results.append({
                'query_id': q_idx,
                'query': query,
                'category': category,
                'system': 'Dense Retrieval',
                'context_window': 256,
                'top_k': 3,
                'latency_ms': latency,
                'memory_available_mb': end_mem,
                'response_length': len(response.get('answer', '')),
                'retrieved_count': len(response.get('retrieved_docs', [])),
                'confidence': response.get('confidence', 0),
                'status': 'success'
            })
            print(f"   Dense: ✅ {latency:.0f}ms | {response.get('answer', '')[:50]}...")
        except Exception as e:
            print(f"   Dense: ❌ {str(e)[:50]}")
            comparison_results.append({
                'query_id': q_idx,
                'query': query,
                'category': category,
                'system': 'Dense Retrieval',
                'context_window': 256,
                'top_k': 3,
                'latency_ms': 0,
                'memory_available_mb': 0,
                'response_length': 0,
                'retrieved_count': 0,
                'confidence': 0,
                'status': 'error'
            })
    
    # System 3: Mobile
    if mobile_available:
        try:
            start = time.time()
            start_mem = psutil.virtual_memory().available / (1024**2)
            
            response = rag_mobile.ask(query, top_k=2)
            
            end = time.time()
            end_mem = psutil.virtual_memory().available / (1024**2)
            latency = (end - start) * 1000
            
            comparison_results.append({
                'query_id': q_idx,
                'query': query,
                'category': category,
                'system': 'Mobile-Optimized',
                'context_window': 128,
                'top_k': 2,
                'latency_ms': latency,
                'memory_available_mb': end_mem,
                'response_length': len(response.get('answer', '')),
                'retrieved_count': len(response.get('retrieved_docs', [])),
                'confidence': response.get('confidence', 0),
                'status': 'success'
            })
            print(f"   Mobile: ✅ {latency:.0f}ms | {response.get('answer', '')[:50]}...")
        except Exception as e:
            print(f"   Mobile: ❌ {str(e)[:50]}")
            comparison_results.append({
                'query_id': q_idx,
                'query': query,
                'category': category,
                'system': 'Mobile-Optimized',
                'context_window': 128,
                'top_k': 2,
                'latency_ms': 0,
                'memory_available_mb': 0,
                'response_length': 0,
                'retrieved_count': 0,
                'confidence': 0,
                'status': 'error'
            })

print("\n" + "="*100)
print("✅ Comparative benchmark complete!")

df_comparison = pd.DataFrame(comparison_results)
print(f"Total results: {len(df_comparison)}")

## 7. System Performance Comparison

In [ ]:
# Group by system
system_stats = df_comparison[df_comparison['status'] == 'success'].groupby('system').agg({
    'latency_ms': ['mean', 'median', 'min', 'max', 'std'],
    'response_length': ['mean', 'min', 'max'],
    'retrieved_count': ['mean', 'min', 'max'],
    'confidence': ['mean', 'std'],
    'memory_available_mb': 'mean',
    'query_id': 'count'
}).round(2)

system_stats.columns = ['Latency_Mean', 'Latency_Med', 'Latency_Min', 'Latency_Max', 'Latency_Std',
                         'Response_Mean', 'Response_Min', 'Response_Max',
                         'Docs_Mean', 'Docs_Min', 'Docs_Max',
                         'Confidence_Mean', 'Confidence_Std',
                         'Memory_MB', 'Queries']

print("\n📊 System Performance Comparison:")
print("="*150)
display(system_stats)

# Speedup comparison (relative to O-RAG)
if orag_available and len(df_comparison[df_comparison['system'] == 'O-RAG']) > 0:
    orag_latency = df_comparison[df_comparison['system'] == 'O-RAG'][df_comparison['status'] == 'success']['latency_ms'].mean()
    
    print("\n⚡ Speedup (relative to O-RAG):")
    for system in df_comparison['system'].unique():
        system_latency = df_comparison[(df_comparison['system'] == system) & 
                                       (df_comparison['status'] == 'success')]['latency_ms'].mean()
        speedup = orag_latency / system_latency if system_latency > 0 else 0
        print(f"  • {system}: {speedup:.2f}x")

## 8. Visualization: System Comparison

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Latency comparison
ax1 = axes[0, 0]
df_comparison[df_comparison['status'] == 'success'].boxplot(column='latency_ms', by='system', ax=ax1)
ax1.set_title('Latency by System', fontsize=12, fontweight='bold')
ax1.set_xlabel('RAG System')
ax1.set_ylabel('Latency (ms)')
plt.sca(ax1)
plt.xticks(rotation=45)

# 2. Response length
ax2 = axes[0, 1]
for system in df_comparison['system'].unique():
    data = df_comparison[df_comparison['system'] == system]['response_length']
    ax2.bar(system, data.mean(), label=system, alpha=0.7)
ax2.set_title('Average Response Length', fontsize=12, fontweight='bold')
ax2.set_ylabel('Characters')
plt.sca(ax2)
plt.xticks(rotation=45)

# 3. Retrieved documents
ax3 = axes[0, 2]
for system in df_comparison['system'].unique():
    data = df_comparison[df_comparison['system'] == system]['retrieved_count']
    ax3.bar(system, data.mean(), label=system, alpha=0.7)
ax3.set_title('Average Retrieved Documents', fontsize=12, fontweight='bold')
ax3.set_ylabel('Count')
plt.sca(ax3)
plt.xticks(rotation=45)

# 4. Confidence scores
ax4 = axes[1, 0]
for system in df_comparison['system'].unique():
    data = df_comparison[df_comparison['system'] == system]['confidence']
    ax4.bar(system, data.mean(), label=system, alpha=0.7, yerr=data.std())
ax4.set_title('Average Confidence Score', fontsize=12, fontweight='bold')
ax4.set_ylabel('Confidence')
ax4.set_ylim([0, 1])
plt.sca(ax4)
plt.xticks(rotation=45)

# 5. Latency vs Confidence scatter
ax5 = axes[1, 1]
colors = {'O-RAG': 'blue', 'Dense Retrieval': 'green', 'Mobile-Optimized': 'red'}
for system in df_comparison['system'].unique():
    data = df_comparison[df_comparison['system'] == system]
    ax5.scatter(data['latency_ms'], data['confidence'], label=system, 
                alpha=0.6, s=100, color=colors.get(system, 'gray'))
ax5.set_title('Latency vs Confidence', fontsize=12, fontweight='bold')
ax5.set_xlabel('Latency (ms)')
ax5.set_ylabel('Confidence')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. Success rate
ax6 = axes[1, 2]
success_rates = df_comparison.groupby('system')['status'].apply(lambda x: (x == 'success').sum() / len(x) * 100)
ax6.bar(success_rates.index, success_rates.values, alpha=0.7, color=['blue', 'green', 'red'])
ax6.set_title('Query Success Rate', fontsize=12, fontweight='bold')
ax6.set_ylabel('Success Rate (%)')
ax6.set_ylim([0, 105])
for i, v in enumerate(success_rates.values):
    ax6.text(i, v + 2, f'{v:.1f}%', ha='center', fontweight='bold')
plt.sca(ax6)
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('rag_systems_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Comparison visualization saved")

## 9. Comprehensive Comparison Table

In [ ]:
# Create detailed comparison table
display_cols = ['query_id', 'query', 'system', 'latency_ms', 'response_length', 
                 'retrieved_count', 'confidence', 'context_window', 'top_k']

df_display = df_comparison[display_cols].copy()
df_display['query'] = df_display['query'].str[:40] + '...'
df_display['latency_ms'] = df_display['latency_ms'].round(2)
df_display['confidence'] = df_display['confidence'].round(3)

print("\n📋 Complete Comparison Results:")
print("="*140)
display(df_display)

## 10. Category-wise Analysis

In [ ]:
# Performance by category
category_perf = df_comparison[df_comparison['status'] == 'success'].pivot_table(
    index='category',
    columns='system',
    values='latency_ms',
    aggfunc='mean'
).round(2)

print("\n📂 Latency by Query Category:")
print("="*70)
display(category_perf)

# Confidence by category
category_conf = df_comparison[df_comparison['status'] == 'success'].pivot_table(
    index='category',
    columns='system',
    values='confidence',
    aggfunc='mean'
).round(3)

print("\n🎯 Confidence by Query Category:")
print("="*70)
display(category_conf)

## 11. Final Recommendation

In [ ]:
# Generate recommendation
recommendation = f"""
<div style='background: #f0f0f0; padding: 20px; border-radius: 10px;'>
  <h2>🎯 RAG System Recommendation</h2>
  
  <h3>📊 Performance Metrics Summary</h3>
  
  <table style='width: 100%; border-collapse: collapse; margin: 10px 0;'>
    <tr style='background: #e0e0e0;'>
      <th style='padding: 8px; text-align: left; border: 1px solid #999;'>Metric</th>
      <th style='padding: 8px; border: 1px solid #999;'>O-RAG</th>
      <th style='padding: 8px; border: 1px solid #999;'>Dense</th>
      <th style='padding: 8px; border: 1px solid #999;'>Mobile</th>
    </tr>
"""

# Add metrics rows
for system in df_comparison['system'].unique():
    system_data = df_comparison[(df_comparison['system'] == system) & 
                                (df_comparison['status'] == 'success')]
    if len(system_data) > 0:
        avg_latency = system_data['latency_ms'].mean()
        avg_conf = system_data['confidence'].mean()
        success_rate = (system_data['status'] == 'success').sum() / len(system_data) * 100

recommendation += f"""
    <tr style='border: 1px solid #999;'>
      <td style='padding: 8px; text-align: left; border: 1px solid #999;'>{system}</td>
      <td style='padding: 8px; border: 1px solid #999;'>{avg_latency:.0f}ms</td>
      <td style='padding: 8px; border: 1px solid #999;'>{avg_conf:.3f}</td>
      <td style='padding: 8px; border: 1px solid #999;'>{success_rate:.1f}%</td>
    </tr>
"""

recommendation += """
  </table>
  
  <h3>✅ Recommendation</h3>
  <ul>
    <li><strong>For General Use:</strong> Use <strong>O-RAG</strong> - Best quality and confidence</li>
    <li><strong>For Speed-Critical:</strong> Use <strong>Mobile-Optimized</strong> - Fastest inference</li>
    <li><strong>For 4GB Devices:</strong> Use <strong>Mobile-Optimized</strong> - Optimized for constraints</li>
    <li><strong>For Balanced:</strong> Use <strong>Dense Retrieval</strong> - Middle ground</li>
  </ul>
</div>
"""

display(HTML(recommendation))